# Dialogue Summarizer — Evaluation on Colab T4

Runs ROUGE evaluation on the DialogSum test split (819 examples).
Compares the fine-tuned LoRA adapter (`rotemso23/dialogsum-phi3-lora`) against the zero-shot baseline.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Add your HuggingFace token in the Colab Secrets tab (key icon, name: `HF_TOKEN`)

**Expected runtime:** ~30–60 minutes (two inference passes over 819 examples).

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Set Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
# Colab already has torch, so we skip it to avoid version conflicts
!pip install -q \
    "datasets>=2.0.0" \
    "transformers>=4.40.0" \
    "peft>=0.19.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.30.0" \
    "rouge-score==0.1.2" \
    "python-dotenv==1.0.1"
print("Dependencies installed.")

## 3. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/rotemso23/dialogue-summarizer.git"
REPO_DIR = "dialogue-summarizer"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## 4. Set HuggingFace token

Your token needs **read** permissions (write not required for evaluation).  
Get one at: https://huggingface.co/settings/tokens

In [ ]:
from google.colab import userdata
import os

# Option A: read from Colab Secrets (Secrets tab on the left sidebar → add HF_TOKEN)
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    # Option B: paste directly (don't commit this)
    os.environ["HF_TOKEN"] = "hf_xxx_YOUR_TOKEN_HERE"
    print("HF_TOKEN set manually — remember not to commit this notebook with a real token.")

# Write to .env so evaluate.py can find it via python-dotenv
with open(".env", "w") as f:
    f.write(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')
print("Token written to .env")

## 5. Run evaluation

This runs two full inference passes over the 819-example test split:
1. Fine-tuned model (`rotemso23/dialogsum-phi3-lora`)
2. Zero-shot baseline (same base model, no adapter)

Results are saved to `evaluation_results.json`.

In [ ]:
!PYTHONPATH=/content/dialogue-summarizer python src/evaluate.py

## 6. View results

In [ ]:
import json

with open("evaluation_results.json") as f:
    results = json.load(f)

print(f"{'Metric':<12} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>10}")
print("-" * 52)
for k in ["rouge1", "rouge2", "rougeL"]:
    base_val = results["baseline"][k]
    ft_val = results["fine_tuned"][k]
    delta = ft_val - base_val
    print(f"{k:<12} {base_val:>10.4f} {ft_val:>12.4f} {delta:>+10.4f}")

## 7. Commit results to GitHub

In [ ]:
!git config user.email "rotemso23@gmail.com"
!git config user.name "rotemso23"

# Requires a GitHub Personal Access Token (classic, repo scope)
# Add it in the Colab Secrets tab with the name GITHUB_TOKEN
try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = "ghp_xxx_YOUR_GITHUB_TOKEN_HERE"

!git add evaluation_results.json
!git commit -m "Add evaluation_results.json (ROUGE scores on DialogSum test split)"
!git push https://{github_token}@github.com/rotemso23/dialogue-summarizer.git master